In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

d:\Git_Repositories\Attention\SelfAttention\.venv\lib\site-packages\torch\_subclasses\functional_tensor.py:279: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:81.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [5]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, k, heads=8):
        super().__init__()

        assert k % heads == 0, "Embedding dimension (k) must be divisible by heads"

        self.k, self.heads = k, heads
        self.s = k // heads

        self.to_queries = nn.Linear(k, k, bias=False)
        self.to_keys = nn.Linear(k, k, bias=False)
        self.to_values = nn.Linear(k, k, bias=False)
        
        # Final linear layer to unify the heads' outputs
        self.unifyheads = nn.Linear(k, k)

    def forward(self, x):
        b, t, k = x.size()
        h = self.heads
        s = self.s

        queries = self.to_queries(x)
        keys = self.to_keys(x)
        values = self.to_values(x)

        # Reshape to split the embedding dimension into h heads
        queries = queries.view(b, t, h, s)
        keys = keys.view(b, t, h, s)
        values = values.view(b, t, h, s)

        # Transpose and fold heads into the batch dimension
        queries = queries.transpose(1, 2).contiguous().view(b * h, t, s)
        keys = keys.transpose(1, 2).contiguous().view(b * h, t, s)
        values = values.transpose(1, 2).contiguous().view(b * h, t, s)

        dot = torch.bmm(queries, keys.transpose(1, 2))  # (b*h, t, t)

        dot = dot / (s ** 0.5)

        attention_weights = F.softmax(dot, dim=2)       # (b*h, t, s)

        output = torch.bmm(attention_weights, values)      # # (b*h, t, s)

        # Reshape and transpose back to original batch/sequence structure
        output = output.view(b, h, t, s)
        output = output.transpose(1, 2).contiguous().view(b, t, k)

        return self.unifyheads(output)



In [6]:
# Parameters
batch_size = 4
seq_length = 10
embedding_dim = 256
num_heads = 8

# Create a random input tensor
x = torch.randn(batch_size, seq_length, embedding_dim)

multi_head_attn = MultiHeadSelfAttention(k=embedding_dim, heads=num_heads)
output_multi = multi_head_attn(x)
print(f"Input shape: {x.shape}")
print(f"Multi-head output shape: {output_multi.shape}")

assert output_multi.shape == x.shape

Input shape: torch.Size([4, 10, 256])
Multi-head output shape: torch.Size([4, 10, 256])
